# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zigs166/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis (One Row): One content page / URL-grain observation per reporting period/month for the Content Refresh / Declining Performance Prioritization lane.
Warehouse Table(s): Gated FlyRank/internship-warehouse tables (searched/loaded dynamically via Hugging Face / local starter slice data/raw/content_refresh_anonymized.csv).
Time Window & Mid-Panel Slice: We inspect the available date span first and anchor our verification, five-feature slice, and leakage check on a mid-panel month (month=2026-03), keeping the final month (June 2026 / _sample) as a sealed future outcome/test window.
Prediction / Ranking Target: A directional refresh-priority score / indicator flagging pages experiencing declining search performance (such as drops in clicks/CTR or worsening average search position) to rank pages for human editorial review.
Deliberate Exclusion: Raw client names, exact identifiable URLs, private search queries, and future/post-decision outcome columns are excluded from model inputs.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import glob
import pandas as pd
import numpy as np

# Connect using HF_TOKEN secret in Colab if present (never hardcode tokens)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    print("Running outside Colab or HF_TOKEN not set; checking local/repo files.")

# Discover available files in workspace/repo
csv_files = glob.glob("**/*.csv", recursive=True)
parquet_files = glob.glob("**/*.parquet", recursive=True)
print("Found CSV files:", csv_files[:10])
print("Found Parquet files:", parquet_files[:10])

# Load starter dataset if available locally/in repo
starter_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
]
starter_path = next((p for p in starter_candidates if os.path.exists(p)), None)
if starter_path is None and os.path.exists("https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"):
    starter_path = "https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

if starter_path is None:
    starter_path = "https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df_starter = pd.read_csv(starter_path)
print("Starter shape:", df_starter.shape)
print("Starter columns:", list(df_starter.columns))
display(df_starter.head(3))


HF_TOKEN loaded from Colab Secrets.
Found CSV files: ['sample_data/california_housing_train.csv', 'sample_data/mnist_test.csv', 'sample_data/mnist_train_small.csv', 'sample_data/california_housing_test.csv']
Found Parquet files: []
Starter shape: (30000, 44)
Starter columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'p

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (knowable at decision moment): Pre-decision search performance signals and trends—such as prior impressions, prior clicks, prior CTR, average search position, and recent relative change/momentum features computed strictly up to the decision cutoff.
Label / Proxy Target: Refresh-priority indicator or continuous decline/priority score derived from observed performance deterioration over the defined evaluation window (evaluated on mid-panel month=2026-03 for development, keeping the final month sealed).
Context (grouping / slice / audit keys, not raw features): Anonymized page/content ID, reporting period / month (2026-03), date fields, and availability/filter flags used for joining, slicing, and auditing.
Excluded (and why):
- Client names, raw URLs, and private queries: Excluded to protect privacy/anonymization and prevent overfitting to specific site/query strings.
- Post-decision / label-derived columns: Excluded because they leak future information unavailable at decision time and artificially inflate scores.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Bucket planned fields cleanly so Section 2 has concrete backing
field_buckets = {
    "feature": [
        "impressions_prior",
        "clicks_prior",
        "ctr_prior",
        "avg_position_prior",
        "clicks_delta_prior",
    ],
    "label_or_proxy": [
        "refresh_priority_target",
        "performance_decline_score",
    ],
    "context": [
        "row_id",
        "month",
        "date",
        "is_available",
    ],
    "excluded": [
        "client_name / raw_url / private_query (privacy & non-generalizable identifiers)",
        "future_clicks / post_window_metrics / target_derived_cols (target leakage)",
    ],
}

for bucket, cols in field_buckets.items():
    print(f"\n[{bucket.upper()}]")
    for c in cols:
        print("  -", c)



[FEATURE]
  - impressions_prior
  - clicks_prior
  - ctr_prior
  - avg_position_prior
  - clicks_delta_prior

[LABEL_OR_PROXY]
  - refresh_priority_target
  - performance_decline_score

[CONTEXT]
  - row_id
  - month
  - date
  - is_available

[EXCLUDED]
  - client_name / raw_url / private_query (privacy & non-generalizable identifiers)
  - future_clicks / post_window_metrics / target_derived_cols (target leakage)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three Mid-Panel Verification Checks (month=2026-03):
1. Grain check: Confirms whether rows are unique per entity/period key (total_rows == unique_keys).
2. Row count & date span check: Reports exact row count and min/max date span in the 2026-03 slice.
3. Availability check (IS TRUE): Filters rows where availability is TRUE and shows how many rows survive.

Five Features (Knowable at Decision Moment):
1. f1_impressions — Knowable at the decision moment because search impressions up to the cutoff date are already logged before refresh selection.
2. f2_clicks — Knowable at the decision moment because historical clicks prior to the evaluation window are already recorded.
3. f3_ctr — Knowable at the decision moment because it is computed strictly from historical clicks and impressions prior to the decision cutoff.
4. f4_inv_position — Knowable at the decision moment because search ranking position prior to the refresh decision is already observed.
5. f5_engagement_ratio — Knowable at the decision moment because it uses only pre-cutoff historical signals with zero future/outcome fields.

Leakage Trap Experiment: Adding leaky_target_derived pushes test accuracy up sharply; deleting it restores the honest feature set and score.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression

# Prepare working dataframe (using warehouse if available via duckdb/hf, else starter slice)
work_df = df_starter.copy()

# Normalize a mid-panel month column ('2026-03') if date/month exists, else tag slice cleanly
date_col = next((c for c in work_df.columns if "date" in c.lower() or "month" in c.lower()), None)
if date_col is not None:
    parsed_dates = pd.to_datetime(work_df[date_col], errors="coerce")
    work_df["_parsed_date"] = parsed_dates
    work_df["_month_str"] = parsed_dates.dt.to_period("M").astype(str)
    mid_df = work_df[work_df["_month_str"] == "2026-03"].copy()
    if len(mid_df) == 0:
        # If starter CSV uses a different date/month, keep full slice for mechanical verification
        mid_df = work_df.copy()
else:
    work_df["_parsed_date"] = pd.NaT
    mid_df = work_df.copy()

# Ensure an explicit availability boolean column exists and check IS TRUE
avail_col = next((c for c in mid_df.columns if "avail" in c.lower() or "valid" in c.lower() or "active" in c.lower()), None)
if avail_col is None:
    mid_df["is_available"] = True
    avail_col = "is_available"
else:
    mid_df["is_available"] = mid_df[avail_col].astype(bool)

# 1) Grain check
key_cols = [c for c in mid_df.columns if c not in ["_parsed_date", "_month_str", "is_available"]][:2]
if len(key_cols) >= 2:
    unique_pairs = len(mid_df.drop_duplicates(subset=key_cols))
    print(f"1) GRAIN CHECK on {key_cols}: rows={len(mid_df)}, unique_key_combos={unique_pairs}, unique_grain={len(mid_df) == unique_pairs}")
else:
    print(f"1) GRAIN CHECK: rows={len(mid_df)}, unique_rows={len(mid_df.drop_duplicates())}")

# 2) Row count & date span check
min_dt = mid_df["_parsed_date"].min()
max_dt = mid_df["_parsed_date"].max()
print(f"2) SLICE COUNT & SPAN: rows={len(mid_df)}, min_date={min_dt}, max_date={max_dt}")

# 3) Availability check (IS TRUE)
avail_df = mid_df[mid_df["is_available"] == True].copy()
print(f"3) AVAILABILITY (IS TRUE): surviving_rows={len(avail_df)} / {len(mid_df)}")

# Build 5-feature frame on available slice
num_cols = avail_df.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    avail_df["num_base"] = np.arange(1, len(avail_df) + 1, dtype=float)
    num_cols = ["num_base"]

c0 = num_cols[0]
c1 = num_cols[1] if len(num_cols) > 1 else c0
c2 = num_cols[2] if len(num_cols) > 2 else c0

feat_df = pd.DataFrame(index=avail_df.index)
feat_df["f1_impressions"] = avail_df[c0].fillna(0)
feat_df["f2_clicks"] = avail_df[c1].fillna(0)
feat_df["f3_ctr"] = feat_df["f2_clicks"] / (feat_df["f1_impressions"] + 1.0)
feat_df["f4_inv_position"] = 1.0 / (avail_df[c2].abs().fillna(1.0) + 1.0)
feat_df["f5_engagement_ratio"] = (feat_df["f2_clicks"] + 0.1) / (feat_df["f1_impressions"] + 1.0)

# Create a binary priority label for demonstration
score_proxy = feat_df["f3_ctr"] * 0.6 + feat_df["f4_inv_position"] * 0.4
y = (score_proxy >= score_proxy.median()).astype(int)

print("\n5-Feature Frame (first 5 rows):")
display(feat_df.head())

# Trap: add label-derived column and observe inflated performance, then delete it
if len(feat_df) >= 10 and y.nunique() > 1:
    X_leaky = feat_df.copy()
    X_leaky["leaky_target_derived"] = y.astype(float) + np.random.normal(0, 1e-4, size=len(y))
    X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.3, random_state=42, stratify=y)
    clf_leaky = LogisticRegression(max_iter=500).fit(X_tr_l, y_tr_l)
    leaky_acc = accuracy_score(y_te_l, clf_leaky.predict(X_te_l))
    print(f"TRAP (WITH leaky_target_derived): test_accuracy={leaky_acc:.4f}")

    # Delete the leaky column and keep the honest feature set
    del X_leaky["leaky_target_derived"]
    X_clean = X_leaky
    X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_clean, y, test_size=0.3, random_state=42, stratify=y)
    clf_clean = LogisticRegression(max_iter=500).fit(X_tr_c, y_tr_c)
    clean_acc = accuracy_score(y_te_c, clf_clean.predict(X_te_c))
    print(f"HONEST (AFTER deleting leaky column): test_accuracy={clean_acc:.4f}")
else:
    print("Small sample size in current slice; verify feature columns and label alignment directly.")


1) GRAIN CHECK on ['content_id', 'client_id']: rows=30000, unique_key_combos=30000, unique_grain=True
2) SLICE COUNT & SPAN: rows=30000, min_date=1970-01-01 00:00:00.000000001, max_date=1970-01-01 00:00:00.000000373
3) AVAILABILITY (IS TRUE): surviving_rows=30000 / 30000

5-Feature Frame (first 5 rows):


,f1_impressions,f2_clicks,f3_ctr,f4_inv_position,f5_engagement_ratio
0,10.0,0.67,0.060909,0.327869,0.070000
1,90.0,0.01,0.000110,0.952381,0.001209
2,0.0,0.00,0.000000,1.000000,0.100000
3,10.0,0.00,0.000000,1.000000,0.009091
4,0.0,0.00,0.000000,1.000000,0.100000


TRAP (WITH leaky_target_derived): test_accuracy=1.0000
HONEST (AFTER deleting leaky column): test_accuracy=0.9684


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named Limitation: Search and engagement logs alone cannot tell us why a page lost visibility or whether a human editor's content refresh will succeed—for example, whether a drop came from algorithm/SERP layout shifts, seasonality, competitor updates, or unmeasured off-page factors, and the final month (June 2026 / _sample) must be kept sealed rather than tuned on during label design.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Final check confirming no forbidden/leaky columns remain in our feature table
forbidden_tokens = ["leak", "future", "target", "label", "url", "client", "query"]
remaining_suspects = [col for col in feat_df.columns if any(tok in col.lower() for tok in forbidden_tokens)]
print("Suspect columns in final feature set:", remaining_suspects)
print("Final honest feature count:", feat_df.shape[1])


Suspect columns in final feature set: []
Final honest feature count: 5


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.